In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

In [3]:
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f+extra)
    return(prob.value,q.value,q_b.value)

In [4]:
def dual (sets,p,R,r,m,r_f,a):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R.dot(a))[i]-(1-sum(a))*r_f - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    obj= cp.Minimize(alpha + beta + gamma * (r-1) + z4 + z2)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,v.value,lbda.value,alpha.value,beta.value,gamma.value,t.value)

In [5]:
np.random.seed(10)

In [50]:
N=100
p = (np.zeros(N)+1)*1/N
I = 1
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
print(R)

[0.04193754]
[[ 0.12069552]
 [ 0.0085441 ]
 [-0.16593948]
 [ 0.02538603]
 [-0.02819644]
 [ 0.30103475]
 [ 0.23942522]
 [-0.15446214]
 [ 0.28343367]
 [-0.06439536]
 [ 0.0766275 ]
 [ 0.29054877]
 [-0.15495059]
 [ 0.08207983]
 [-0.17609506]
 [-0.33825994]
 [ 0.24731956]
 [ 0.05564547]
 [-0.11447352]
 [-0.26731035]
 [-0.03029456]
 [ 0.31843586]
 [ 0.13049382]
 [-0.02455228]
 [-0.08440469]
 [ 0.07116242]
 [-0.25946108]
 [ 0.3189613 ]
 [ 0.15063784]
 [ 0.28887012]
 [-0.06270113]
 [ 0.22096575]
 [ 0.18758095]
 [-0.25661374]
 [ 0.07979214]
 [ 0.11302223]
 [-0.06498353]
 [-0.02203423]
 [ 0.13723706]
 [ 0.04241642]
 [-0.15094327]
 [ 0.16829062]
 [ 0.09556347]
 [ 0.09983573]
 [ 0.06915265]
 [ 0.13980435]
 [-0.0072004 ]
 [-0.12255166]
 [-0.09836497]
 [ 0.27111503]
 [-0.35354373]
 [ 0.15810814]
 [-0.23845978]
 [-0.27177007]
 [-0.15131371]
 [-0.00150672]
 [ 0.19610149]
 [-0.2896803 ]
 [ 0.38481511]
 [ 0.28274474]
 [ 0.02348528]
 [-0.00804916]
 [-0.14070648]
 [ 0.16760812]
 [ 0.06376029]
 [ 0.3324128

In [64]:
a = np.zeros(I)+1/I
r = 0.043
m = 0.95
r_f = 0.001
robustcheck(a,R,r,p,m,r_f)

0.42911440258724753


(0.9429398661127437,
 array([0.00959587, 0.00959383, 0.00957277, 0.00959544, 0.00959005,
        0.00963933, 0.00962379, 0.00957442, 0.00963471, 0.00958781,
        0.00959922, 0.00963902, 0.0095738 , 0.00959965, 0.00957238,
        0.00957208, 0.00962663, 0.00959789, 0.00958093, 0.00957196,
        0.00958933, 0.00964113, 0.00959571, 0.00959074, 0.00958535,
        0.009599  , 0.00957186, 0.00963926, 0.00959347, 0.00963809,
        0.00958859, 0.00960519, 0.00959604, 0.0095718 , 0.00959943,
        0.00959743, 0.00958701, 0.00959142, 0.00959551, 0.00959594,
        0.00957584, 0.00959136, 0.00960032, 0.00960035, 0.00959875,
        0.00959437, 0.00959267, 0.00957829, 0.00958272, 0.00963064,
        0.00956938, 0.00959261, 0.0095718 , 0.00957209, 0.00957511,
        0.00959326, 0.00959989, 0.00957224, 0.00960633, 0.00963256,
        0.00959491, 0.00959206, 0.00957743, 0.00959142, 0.00959819,
        0.00963706, 0.00958448, 0.0095919 , 0.00959648, 0.00957915,
        0.00962217, 0.00959

In [101]:

x=np.arange(1,N)
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
sets =psets

In [80]:
dual (sets,p,R,r,m,r_f,a)

(0.8233203712823867,
 array([[-4.85900015e-11, -5.08273252e-11, -4.99037147e-11,
         -5.08016132e-11],
        [-5.06601201e-11, -4.00770563e-11, -0.00000000e+00,
         -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00, -4.88070569e-11,
         -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
         -4.97513453e-11],
        [-4.56207618e-11, -4.58189950e-11, -0.00000000e+00,
         -0.00000000e+00],
        [-4.54629898e-11, -0.00000000e+00, -4.45116437e-11,
         -0.00000000e+00],
        [-4.54361154e-11, -0.00000000e+00, -0.00000000e+00,
         -4.56380042e-11],
        [-0.00000000e+00, -4.56740133e-11, -4.44988595e-11,
         -0.00000000e+00],
        [-0.00000000e+00, -4.56275880e-11, -0.00000000e+00,
         -4.56101813e-11],
        [-0.00000000e+00, -0.00000000e+00, -4.47433728e-11,
         -4.58743639e-11],
        [-4.74381588e-11, -4.76255202e-11, -4.65610347e-11,
         -0.00000000e+00],
        [-4.74000374e-

In [54]:
[probv,vv,lbdav,alphav,betav,gammav,tv]=dual (sets,p,R,r,m,r_f,a)
N = len(p)
M = len(sets)
cons1 =np.zeros(N)
cons2 = np.zeros(N)
z0 = 0
for j in range(M):
    z9 = -np.min(vv[j,sets[j]])*(1-m)+lbdav[j]
    z0 = z0 + max(z9,0)
for i in range(N):
    lbdsom = 0
    for j in range(M):
        if i in sets[j]:
            lbdsom = lbdsom + lbdav[j]
    cons1[i] = R.dot(a)[i] + betav + lbdsom
    cons2[i] = gammav * np.exp((-alphav+sum(vv[0:M:1,i]))/gammav)-tv[i]
print(cons1)
print(cons2)
print(-1+alphav+betav+gammav*r+sum(p*tv)+z0)

[ 5.70003067e-09 -5.38403810e-09  6.53539289e+00  1.70622106e+01]
[-8.10329546e-08 -2.98001260e-06 -3.94966149e-08 -2.06583066e-08]
[24.65265508]
